# Amazon Review 2023
Amazon Reviews dataset is large-scale dataset collected in 2023 by McAuley Lab, and it includes rich features such as:
- User Reviews (ratings, text, helpfulness votes, etc.);
- Item Metadata (descriptions, price, raw image, etc.);
- Links (user-item / bought together graphs).

Related Information
- HP: https://amazon-reviews-2023.github.io/
- paper: [Bridging Language and Items for Retrieval and Recommendation](https://arxiv.org/abs/2403.03952)


In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

import pathlib

import torch_geometric.transforms as T

from ml_sandbox_libs.data.amazon_reviews_dataset import (
    AmazonReviewsBipartiteGraphDataModule,
    AmazonReviewsSeqRecDataModule,
    fetch_dataset,
    fetch_metadata,
    to_bipartite_graph_batch,
)
from ml_sandbox_libs.data.amazon_reviews_dataset.bipartite_graph import (
    bipartite_graph_preprocess_dataset,
    create_bipartite_graph,
)

In [3]:
dataset_dict = fetch_dataset(category="Video_Games", dataset_type="0core_timestamp_w_his")
df = dataset_dict["train"].to_polars()
df.head()

2026-04-14 20:38:01.376 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_dataset:41 - Fetching Amazon Reviews 2023 dataset


user_id,parent_asin,rating,timestamp,history
str,str,str,str,str
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07SRWRH5D""","""5.0""","""1587051114941""",""""""
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07DK1H3H5""","""4.0""","""1608186804795""","""B07SRWRH5D"""
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""","""B07MFMFW34""","""5.0""","""1490877431000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B00HUWA45W""","""5.0""","""1427591932000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B0BCHWZX95""","""5.0""","""1577637634017""","""B00HUWA45W"""


In [4]:
print("Dataset Size")
print(
    f"train: {len(dataset_dict['train'])}, valid: {len(dataset_dict['valid'])}, test: {len(dataset_dict['test'])}"
)

Dataset Size
train: 3847041, valid: 344592, test: 363867


The dataset schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews

| Field            | Type     | Explanation                                                                                                                                                                               |
|------------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| user_id          | str      | ID of the reviewer                                                                                                                                                                       |
| parent_asin      | str      | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. **Please use parent ID to find product meta.** |
| rating           | float    | Rating of the product (from 1.0 to 5.0).                                                                                                                                                   |
| timestamp        | int      | Time of the review (unix time)                                                                                                                                                           |
| history | str     | parent_asin list which was bought by user before. The separator is ' '                                                                                                                                                               |

In [5]:
metadata_dataset = fetch_metadata(category="Video_Games")
metadata_df = metadata_dataset.to_polars().select(
    ["parent_asin", "title", "categories", "main_category", "average_rating", "rating_number"]
)
metadata_df.head(5)

2026-04-14 20:38:04.573 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_metadata:61 - Fetching Amazon Reviews 2023 metadata


parent_asin,title,categories,main_category,average_rating,rating_number
str,str,list[str],str,f64,i64
"""B000FH0MHO""","""Dash 8-300 Professional Add-On""","[""Video Games"", ""PC"", ""Games""]","""Video Games""",5.0,1
"""B00069EVOG""","""Phantasmagoria: A Puzzle of Fl…","[""Video Games"", ""PC"", ""Games""]","""Video Games""",4.1,18
"""B00Z9TLVK0""","""NBA 2K17 - Early Tip Off Editi…","[""Video Games"", ""PlayStation 4"", ""Games""]","""Video Games""",4.3,223
"""B07SZJZV88""","""Nintendo Selects: The Legend o…","[""Video Games"", ""Legacy Systems"", … ""Games""]","""Video Games""",4.9,22
"""B002WH4ZJG""","""Thrustmaster Elite Fitness Pac…","[""Video Games"", ""Legacy Systems"", … ""Fitness Accessories""]","""Video Games""",3.0,3


In [6]:
print(f"Parent Asin Size: {len(metadata_df)}")

Parent Asin Size: 137269


The metadata schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata

| Field           | Type   | Explanation                                                                                                                                                             |
|-----------------|--------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| parent_asin     | str    | Parent ID of the product.                                                                                                                                                |
| title           | str    | Name of the product.                                                                                                                                                     |
| categories      | list   | Hierarchical categories of the product.                                                                                                                                  |


In [7]:
datamodule = AmazonReviewsSeqRecDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=4,
    max_seq_len=5,
    neg_sample_size=2,
    sampling_val_test=True,
    eval_negative_sample_size=10,
    filter_no_history=False,
)

datamodule.prepare_data()
datamodule.setup(stage="fit")
train_dataloader = datamodule.train_dataloader()
batch = next(iter(train_dataloader))

2026-04-14 20:38:06.318 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:prepare_data:534 - Loading preprocessed dataset
2026-04-14 20:38:08.315 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:setup:614 - 
        Train Dataset: 3847041
        Val Dataset: 100000
        User2Index: 2149656
        Item2Index: 34154
        Category2Index: 194
        


In [8]:
datamodule.train_df.head(5)

user_id,user_index,parent_asin,item_index,category,category_index,average_rating,rating_number,rating,timestamp,history,history_index,history_category,history_category_index,history_average_rating,history_rating_number
str,i64,str,i64,str,i64,f64,i64,f64,i64,list[str],list[i64],list[str],list[i64],list[f64],list[i64]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216274,"""B07SRWRH5D""",30007,"""Video Games/PlayStation 4/Game…",164,4.8,9097,5.0,1587051114941,[],[],[],[],[],[]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216274,"""B07DK1H3H5""",27747,"""Video Games/PC/Games""",146,4.1,2015,4.0,1608186804795,"[""B07SRWRH5D""]",[30007],"[""Video Games/PlayStation 4/Games""]",[164],[4.8],[9097]
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""",0,"""B07MFMFW34""",29082,"""Video Games/PC/Games""",146,3.0,31,5.0,1490877431000,[],[],[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961140,"""B00HUWA45W""",19294,"""Video Games/Xbox One/Accessori…",177,4.0,287,5.0,1427591932000,[],[],[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961140,"""B0BCHWZX95""",33732,"""Video Games/Nintendo Switch/Ac…",121,4.6,19492,5.0,1577637634017,"[""B00HUWA45W""]",[19294],"[""Video Games/Xbox One/Accessories""]",[177],[4.0],[287]


In [9]:
batch.user_index, batch.pos_item_index, batch.neg_item_indexes, batch.item_history

(tensor([1183051, 2034309]),
 tensor([1, 1]),
 tensor([[21179,  5833],
         [17200, 20326]]),
 tensor([[    0,     0,     0,     0, 19763],
         [    0,     0,     0,     0,     0]]))

## Bipartite Graph

In [10]:
(
    all_df,
    user2index,
    item2index,
    category2index,
    item_index_2_category_index,
) = bipartite_graph_preprocess_dataset(dataset_dict=dataset_dict, metadata=metadata_dataset)

2026-04-14 20:39:09.485 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.bipartite_graph:bipartite_graph_preprocess_dataset:116 - Preprocessing the dataset for bipartite graph


In [11]:
all_df.head(5)

split,user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,num_ratings
str,str,i64,str,i64,str,i64,f64,i64,u32
"""train""","""AGC27XQGMTG6RZZMVLGDZQM4X26A""",0,"""B000LWDCO8""",7428,"""Video Games/Legacy Systems/Nin…",41,3.0,1260110346000,1
"""train""","""AHVZEEL5CZJFCZYEAQ5PQAIWYDGA""",2081977,"""B00005NOFH""",2934,"""Video Games/Legacy Systems/Pla…",84,5.0,1369255075000,1
"""train""","""AGMJO42N3UQFUOOXMZTTH5XMM73Q""",1384558,"""B00JFCA6V2""",19620,"""Video Games/Legacy Systems/Pla…",80,5.0,1505330080559,1
"""train""","""AETCM2H45G5I2KTJ3VBVD56RB4QA""",423563,"""B000BPLCDI""",6351,"""Video Games/Legacy Systems/Xbo…",104,4.0,1383547644000,1
"""train""","""AE5DE7F2DBVR64ZRR2IL2XAW3WNQ""",55412,"""B00Z9LUDX4""",21670,"""Video Games/PlayStation 4/Game…",164,5.0,1471143919000,1


In [12]:
train_data = create_bipartite_graph(
    split="train",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)
val_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)

transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
train_data = transform(train_data)
val_data = transform(val_data)

/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/src/ml_sandbox_libs/data/amazon_reviews_dataset/bipartite_graph.py:208: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  edge_attr = torch.as_tensor(


In [13]:
train_data

HeteroData(
  user={
    x=[2149656, 1],
    user_index=[2149656],
  },
  item={
    x=[34153, 1],
    item_index=[34153],
    category_index=[34153],
  },
  (user, rates, item)={
    edge_index=[2, 3847041],
    edge_attr=[3847041, 1],
    edge_label_index=[2, 3847041],
    edge_label_attr=[3847041, 1],
  },
  (item, rated_by, user)={
    edge_index=[2, 3847041],
    edge_attr=[3847041, 1],
  }
)

In [14]:
# daloaloader

datamodule = AmazonReviewsBipartiteGraphDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=1,
    neg_sample_size=5,
    sampling_val_test=False,
    eval_negative_sample_size=10,
)
datamodule.prepare_data()
datamodule.setup(stage="fit")

2026-04-14 20:39:16.116 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_dataset:41 - Fetching Amazon Reviews 2023 dataset
2026-04-14 20:39:17.661 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_metadata:61 - Fetching Amazon Reviews 2023 metadata
2026-04-14 20:39:25.544 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.bipartite_graph:bipartite_graph_preprocess_dataset:116 - Preprocessing the dataset for bipartite graph


In [15]:
train_dataloader = datamodule.train_dataloader()

In [16]:
batch = next(iter(train_dataloader))
batch

HeteroData(
  user={
    x=[138, 1],
    user_index=[138],
    n_id=[138],
    num_sampled_nodes=[3],
    src_index=[2],
  },
  item={
    x=[192, 1],
    item_index=[192],
    category_index=[192],
    n_id=[192],
    num_sampled_nodes=[3],
    dst_pos_index=[2],
    dst_neg_index=[2, 5],
  },
  (user, rates, item)={
    edge_index=[2, 138],
    edge_attr=[138, 1],
    edge_label_attr=[138, 1],
    e_id=[138],
    num_sampled_edges=[2],
    input_id=[2],
  },
  (item, rated_by, user)={
    edge_index=[2, 305],
    edge_attr=[305, 1],
    e_id=[305],
    num_sampled_edges=[2],
  }
)

In [17]:
batch["user"].n_id

tensor([1603317, 1930915, 1640084, 1891022,  110536,  731377, 1776700, 1540335,
        1621429,  389474,  194808,  411257,  803152, 1144810, 1450297,  965097,
         749227,  376791, 2054370, 1323186, 1688516,  540903,  692884,  200071,
         249889,  192664, 2042685,  707012,  902036,  864559, 1230640,  181159,
         246799,  531629, 1283135,   75569,  514455, 1055368, 1804551, 1337457,
        1033321,  913692,  308817,  527593, 2041847, 1593894, 1711054, 1921979,
        1289324,  200302, 2024093,  231239,  478569, 1451845,  827441, 1828116,
        1923732,   45105, 1328878, 1883893,  407476,       0, 1894118, 1831115,
        1734678, 1566365,  986867,  922828, 1415897, 1443211,  648783, 2091607,
         126171, 1042262, 1199270,  320667,  871909, 1945728, 1488421,  590631,
        1547506,  462704, 1932526, 1477867,  592024, 2018100,  247240, 1223032,
        1664882, 1324343, 1931550,  839879, 1719922,  399954, 1492132, 1469518,
         167722, 1792291, 1692813, 16665

In [18]:
batch["user"].user_index

tensor([1603317, 1930915, 1640084, 1891022,  110536,  731377, 1776700, 1540335,
        1621429,  389474,  194808,  411257,  803152, 1144810, 1450297,  965097,
         749227,  376791, 2054370, 1323186, 1688516,  540903,  692884,  200071,
         249889,  192664, 2042685,  707012,  902036,  864559, 1230640,  181159,
         246799,  531629, 1283135,   75569,  514455, 1055368, 1804551, 1337457,
        1033321,  913692,  308817,  527593, 2041847, 1593894, 1711054, 1921979,
        1289324,  200302, 2024093,  231239,  478569, 1451845,  827441, 1828116,
        1923732,   45105, 1328878, 1883893,  407476,       0, 1894118, 1831115,
        1734678, 1566365,  986867,  922828, 1415897, 1443211,  648783, 2091607,
         126171, 1042262, 1199270,  320667,  871909, 1945728, 1488421,  590631,
        1547506,  462704, 1932526, 1477867,  592024, 2018100,  247240, 1223032,
        1664882, 1324343, 1931550,  839879, 1719922,  399954, 1492132, 1469518,
         167722, 1792291, 1692813, 16665

In [19]:
batch[("user", "rates", "item")].edge_index

tensor([[  2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,
          16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,
          30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,
          44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,
          58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  61,  69,  70,
          71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,
          85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,
          99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
         113, 114, 115, 116,  61, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   2,   2,   2,   2,   2,   2,   2,   2,
           2,   2,   3,   3,   3,   3,   3,   3,   3,   3,   

In [20]:
batch["item"].item_index

tensor([ 2153,  2266,  4766,  8155,  9440, 10860, 22828, 22926, 23160, 25764,
        30935, 33167, 26128, 15885, 14524, 30568, 10619,   552,  1740,  2342,
        32935,   152,  5798, 10310,  7700,  6451, 18859,  6693, 27393, 21460,
         2262,  2180,  2997,   198,  1760,  9538,   299,   386,  8059,     1,
        27725, 11560,  3201,  3641,  9676,  9937,  2911, 22643,  7809,  4158,
           72,   805,   205,  2581,  1684,  2046,   157,   890,  2206, 15864,
          635, 13786,  9324,  5052, 13040, 16624,  4086,  6776,  7355,  3561,
         9695,  7004,  9075, 10501, 19694, 22502,  2765,  7345,  8117, 28575,
         8250,  6040, 10339,  6584, 10369, 10514, 10230,  9923,  6725, 12842,
        10248, 33214, 10828, 18301, 30305, 14913, 12962, 12347,  1167, 28672,
        15959,  4467,  6659, 12820,  9399, 10315, 30840, 26099,  2586, 29587,
        16337, 10610,  7749,  7562, 23426, 18088, 22425, 13687, 21966, 31160,
        18180, 23534, 21867, 19427, 31651, 32708, 16290, 23775, 

In [21]:
batch["item"].n_id

tensor([ 2153,  2266,  4766,  8155,  9440, 10860, 22828, 22926, 23160, 25764,
        30935, 33167, 26128, 15885, 14524, 30568, 10619,   552,  1740,  2342,
        32935,   152,  5798, 10310,  7700,  6451, 18859,  6693, 27393, 21460,
         2262,  2180,  2997,   198,  1760,  9538,   299,   386,  8059,     1,
        27725, 11560,  3201,  3641,  9676,  9937,  2911, 22643,  7809,  4158,
           72,   805,   205,  2581,  1684,  2046,   157,   890,  2206, 15864,
          635, 13786,  9324,  5052, 13040, 16624,  4086,  6776,  7355,  3561,
         9695,  7004,  9075, 10501, 19694, 22502,  2765,  7345,  8117, 28575,
         8250,  6040, 10339,  6584, 10369, 10514, 10230,  9923,  6725, 12842,
        10248, 33214, 10828, 18301, 30305, 14913, 12962, 12347,  1167, 28672,
        15959,  4467,  6659, 12820,  9399, 10315, 30840, 26099,  2586, 29587,
        16337, 10610,  7749,  7562, 23426, 18088, 22425, 13687, 21966, 31160,
        18180, 23534, 21867, 19427, 31651, 32708, 16290, 23775, 

In [22]:
import torch

torch.equal(batch["item"].n_id, batch["item"].item_index)

True

In [23]:
batch

HeteroData(
  user={
    x=[138, 1],
    user_index=[138],
    n_id=[138],
    num_sampled_nodes=[3],
    src_index=[2],
  },
  item={
    x=[192, 1],
    item_index=[192],
    category_index=[192],
    n_id=[192],
    num_sampled_nodes=[3],
    dst_pos_index=[2],
    dst_neg_index=[2, 5],
  },
  (user, rates, item)={
    edge_index=[2, 138],
    edge_attr=[138, 1],
    edge_label_attr=[138, 1],
    e_id=[138],
    num_sampled_edges=[2],
    input_id=[2],
  },
  (item, rated_by, user)={
    edge_index=[2, 305],
    edge_attr=[305, 1],
    e_id=[305],
    num_sampled_edges=[2],
  }
)

In [26]:
typed_batch = to_bipartite_graph_batch(batch)

In [30]:
typed_batch

AmazonReviewsBipartiteGraphBatch(user_node_ids=tensor([1603317, 1930915, 1640084, 1891022,  110536,  731377, 1776700, 1540335,
        1621429,  389474,  194808,  411257,  803152, 1144810, 1450297,  965097,
         749227,  376791, 2054370, 1323186, 1688516,  540903,  692884,  200071,
         249889,  192664, 2042685,  707012,  902036,  864559, 1230640,  181159,
         246799,  531629, 1283135,   75569,  514455, 1055368, 1804551, 1337457,
        1033321,  913692,  308817,  527593, 2041847, 1593894, 1711054, 1921979,
        1289324,  200302, 2024093,  231239,  478569, 1451845,  827441, 1828116,
        1923732,   45105, 1328878, 1883893,  407476,       0, 1894118, 1831115,
        1734678, 1566365,  986867,  922828, 1415897, 1443211,  648783, 2091607,
         126171, 1042262, 1199270,  320667,  871909, 1945728, 1488421,  590631,
        1547506,  462704, 1932526, 1477867,  592024, 2018100,  247240, 1223032,
        1664882, 1324343, 1931550,  839879, 1719922,  399954, 1492132, 14